In [ ]:
import pandas as pd
import numpy as np

# 1. Load Dataset
file_path = '/kaggle/input/datasets/muhammadzakikurnia/fitness/fitness.jsonl'
df = pd.read_json(file_path, lines=True)

print(f"Total Baris Awal: {len(df)}")

# Buang kalimat "I don't have data on that"
kalimat_sampah = "I don't have data on that"
df = df[~df['answer'].astype(str).str.contains(kalimat_sampah, case=False, na=False)]

# 3. Filter Tambahan: Buang jawaban yang terlalu pendek (< 15 karakter)
df = df[df['answer'].astype(str).str.len() >= 15]

# 4. Hapus sisa NaN
df_bersih = df.dropna(subset=['question', 'answer'])

print(f"Total Baris Setelah Dibersihkan: {len(df_bersih)}")

# 5. Ambil Sampel 10.000 Baris untuk Training
df_final = df_bersih.sample(n=10000, random_state=42)

print(f"Total Baris Siap Training (Sampel): {len(df_final)}")

# 6. Simpan File ke Folder Output Kaggle
output_path = '/kaggle/working/dataset_fitness_bersih_10k.csv'
df_final.to_csv(output_path, index=False)
print(f"Dataset final berhasil disimpan di: {output_path}")

In [ ]:
# 1. Load data CSV yang sudah kita bersihkan tadi
print("Memuat dataset...")
df = pd.read_csv('/kaggle/working/dataset_fitness_bersih_10k.csv')
dataset = Dataset.from_pandas(df)

# 2. tokenize
model_name = "microsoft/DialoGPT-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

tokenizer.pad_token = tokenizer.eos_token

# 3. Fungsi untuk menggabungkan dan men-tokenisasi teks
def preprocess_function(examples):
    inputs = [q + tokenizer.eos_token + a + tokenizer.eos_token for q, a in zip(examples['question'], examples['answer'])]

    tokenized_inputs = tokenizer(inputs, max_length=128, truncation=True, padding="max_length")
    return tokenized_inputs

print("Melakukan tokenisasi data...")
# Terapkan preprocessing ke seluruh dataset
tokenized_dataset = dataset.map(preprocess_function, batched=True, remove_columns=dataset.column_names)
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)